# ydata-profiling → SQL Server Metadata Pipeline
Extracts profiling results and stores them in SQL Server across 4 tables:
- `metadata_dataset_overview` — row/column counts, duplicates, missing %
- `metadata_column_stats` — type, missing count/%, unique count
- `metadata_descriptive_stats` — mean, std, min, max, quartiles (numeric columns)
- `metadata_correlations` — pairwise Pearson correlations

## 1. Install dependencies

In [ ]:
# Run once if not already installed
# !pip install ydata-profiling pyodbc sqlalchemy pandas

## 2. Imports

In [ ]:
import pandas as pd
from ydata_profiling import ProfileReport
from sqlalchemy import create_engine, text
import urllib
from datetime import datetime

## 3. SQL Server connection
Choose **Option A** (Windows Auth) or **Option B** (Username & Password) and comment out the other.

In [ ]:
# ── CONFIGURE THESE ──────────────────────────────────────────────────────────
SQL_SERVER   = "YOUR_SERVER_NAME"       # e.g. 'localhost' or 'myserver.database.windows.net'
SQL_DATABASE = "YOUR_DATABASE_NAME"
SQL_SCHEMA   = "dbo"                    # change if needed

# Option A: Windows Authentication (trusted connection) ----------------------
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={SQL_SERVER};DATABASE={SQL_DATABASE};"
    f"Trusted_Connection=yes;"
)

# Option B: Username & Password -----------------------------------------------
# SQL_USERNAME = "YOUR_USERNAME"
# SQL_PASSWORD = "YOUR_PASSWORD"
# params = urllib.parse.quote_plus(
#     f"DRIVER={{ODBC Driver 17 for SQL Server}};"
#     f"SERVER={SQL_SERVER};DATABASE={SQL_DATABASE};"
#     f"UID={SQL_USERNAME};PWD={SQL_PASSWORD};"
# )

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
print("Engine created. Testing connection...")
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print("Connection successful.")

## 4. Load your data
Replace the sample DataFrame below with your own data source.

In [ ]:
# ── REPLACE WITH YOUR DATA ────────────────────────────────────────────────────
# df = pd.read_csv("your_file.csv")
# df = pd.read_excel("your_file.xlsx")
# df = pd.read_sql("SELECT * FROM your_table", engine)

# Sample data for demonstration
df = pd.DataFrame({
    "age":    [25, 32, None, 45, 28],
    "salary": [50000, 62000, 71000, None, 48000],
    "dept":   ["HR", "Eng", "Eng", "HR", "Fin"],
    "tenure": [2.5, 5.0, 3.1, 9.0, 1.2]
})

DATASET_NAME = "sample_employees"    # logical name stored in the metadata tables
RUN_TS = datetime.utcnow()           # timestamp for this profiling run

df.head()

## 5. Run ydata-profiling

In [ ]:
profile = ProfileReport(df, title=DATASET_NAME, explorative=True)

# Optional: view inline
# profile.to_notebook_iframe()

## 6. Extract metadata from profile

In [ ]:
desc = profile.get_description()

# ── 6a. Dataset overview ─────────────────────────────────────────────────────
ov = desc.analysis  # dataset-level stats
table_stats = desc.table

overview_df = pd.DataFrame([{
    "dataset_name":        DATASET_NAME,
    "run_timestamp":       RUN_TS,
    "n_rows":              int(table_stats.get("n", 0)),
    "n_cols":              int(table_stats.get("n_var", 0)),
    "n_missing_cells":     int(table_stats.get("n_cells_missing", 0)),
    "pct_missing_cells":   round(float(table_stats.get("p_cells_missing", 0)) * 100, 4),
    "n_duplicate_rows":    int(table_stats.get("n_duplicates", 0)),
    "pct_duplicate_rows":  round(float(table_stats.get("p_duplicates", 0)) * 100, 4),
    "n_numeric_cols":      int(table_stats.get("types", {}).get("Numeric", 0)),
    "n_categorical_cols":  int(table_stats.get("types", {}).get("Categorical", 0)),
}])

print("Dataset overview:")
overview_df

In [ ]:
# ── 6b. Column stats + descriptive stats ─────────────────────────────────────
col_stats_rows = []
desc_stats_rows = []

for col_name, col_data in desc.variables.items():
    col_type = str(col_data.get("type", "Unknown"))
    n_rows   = int(table_stats.get("n", 1))

    col_stats_rows.append({
        "dataset_name":  DATASET_NAME,
        "run_timestamp": RUN_TS,
        "column_name":   col_name,
        "column_type":   col_type,
        "n_missing":     int(col_data.get("n_missing", 0)),
        "pct_missing":   round(float(col_data.get("p_missing", 0)) * 100, 4),
        "n_unique":      int(col_data.get("n_unique", col_data.get("n_distinct", 0))),
        "pct_unique":    round(int(col_data.get("n_unique", col_data.get("n_distinct", 0))) / n_rows * 100, 4),
    })

    # Only extract descriptive stats for numeric columns
    if col_type in ("Numeric", "NUM") or col_data.get("mean") is not None:
        desc_stats_rows.append({
            "dataset_name":  DATASET_NAME,
            "run_timestamp": RUN_TS,
            "column_name":   col_name,
            "mean":          round(float(col_data.get("mean", 0) or 0), 6),
            "std":           round(float(col_data.get("std",  0) or 0), 6),
            "min":           round(float(col_data.get("min",  0) or 0), 6),
            "pct_25":        round(float(col_data.get("25%",  0) or 0), 6),
            "median":        round(float(col_data.get("50%",  0) or 0), 6),
            "pct_75":        round(float(col_data.get("75%",  0) or 0), 6),
            "max":           round(float(col_data.get("max",  0) or 0), 6),
            "kurtosis":      round(float(col_data.get("kurtosis", 0) or 0), 6),
            "skewness":      round(float(col_data.get("skewness", 0) or 0), 6),
        })

col_stats_df  = pd.DataFrame(col_stats_rows)
desc_stats_df = pd.DataFrame(desc_stats_rows)

print("Column stats:")
display(col_stats_df)
print("\nDescriptive stats:")
display(desc_stats_df)

In [ ]:
# ── 6c. Correlations ──────────────────────────────────────────────────────────
corr_rows = []

if desc.correlations:
    # Use Pearson if available, else first available method
    corr_key = "pearson" if "pearson" in desc.correlations else next(iter(desc.correlations), None)
    if corr_key:
        corr_matrix = desc.correlations[corr_key]
        for col_a in corr_matrix.columns:
            for col_b in corr_matrix.columns:
                if col_a < col_b:   # upper triangle only — avoids duplicates
                    val = corr_matrix.loc[col_a, col_b] if col_a in corr_matrix.index else None
                    if val is not None and not pd.isna(val):
                        corr_rows.append({
                            "dataset_name":      DATASET_NAME,
                            "run_timestamp":     RUN_TS,
                            "correlation_type":  corr_key,
                            "column_a":          col_a,
                            "column_b":          col_b,
                            "correlation_value": round(float(val), 6),
                        })

corr_df = pd.DataFrame(corr_rows)
print("Correlations:")
corr_df

## 7. Create SQL Server tables (auto-create)

In [ ]:
DDL_STATEMENTS = [

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'metadata_dataset_overview')
CREATE TABLE [{SQL_SCHEMA}].[metadata_dataset_overview] (
    id                   INT IDENTITY(1,1) PRIMARY KEY,
    dataset_name         NVARCHAR(255)  NOT NULL,
    run_timestamp        DATETIME2      NOT NULL,
    n_rows               INT,
    n_cols               INT,
    n_missing_cells      INT,
    pct_missing_cells    FLOAT,
    n_duplicate_rows     INT,
    pct_duplicate_rows   FLOAT,
    n_numeric_cols       INT,
    n_categorical_cols   INT
);""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'metadata_column_stats')
CREATE TABLE [{SQL_SCHEMA}].[metadata_column_stats] (
    id               INT IDENTITY(1,1) PRIMARY KEY,
    dataset_name     NVARCHAR(255)  NOT NULL,
    run_timestamp    DATETIME2      NOT NULL,
    column_name      NVARCHAR(255)  NOT NULL,
    column_type      NVARCHAR(100),
    n_missing        INT,
    pct_missing      FLOAT,
    n_unique         INT,
    pct_unique       FLOAT
);""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'metadata_descriptive_stats')
CREATE TABLE [{SQL_SCHEMA}].[metadata_descriptive_stats] (
    id               INT IDENTITY(1,1) PRIMARY KEY,
    dataset_name     NVARCHAR(255)  NOT NULL,
    run_timestamp    DATETIME2      NOT NULL,
    column_name      NVARCHAR(255)  NOT NULL,
    mean             FLOAT,
    std              FLOAT,
    min              FLOAT,
    pct_25           FLOAT,
    median           FLOAT,
    pct_75           FLOAT,
    max              FLOAT,
    kurtosis         FLOAT,
    skewness         FLOAT
);""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'metadata_correlations')
CREATE TABLE [{SQL_SCHEMA}].[metadata_correlations] (
    id                  INT IDENTITY(1,1) PRIMARY KEY,
    dataset_name        NVARCHAR(255)  NOT NULL,
    run_timestamp       DATETIME2      NOT NULL,
    correlation_type    NVARCHAR(50),
    column_a            NVARCHAR(255)  NOT NULL,
    column_b            NVARCHAR(255)  NOT NULL,
    correlation_value   FLOAT
);"""
]

with engine.begin() as conn:
    for stmt in DDL_STATEMENTS:
        conn.execute(text(stmt))

print("All tables created (or already exist).")

## 8. Insert metadata into SQL Server

In [ ]:
TABLE_MAP = {
    "metadata_dataset_overview":   overview_df,
    "metadata_column_stats":       col_stats_df,
    "metadata_descriptive_stats":  desc_stats_df,
    "metadata_correlations":       corr_df,
}

with engine.begin() as conn:
    for table_name, df_to_insert in TABLE_MAP.items():
        if df_to_insert.empty:
            print(f"  Skipped {table_name} (no data).")
            continue
        df_to_insert.to_sql(
            name=table_name,
            con=conn,
            schema=SQL_SCHEMA,
            if_exists="append",   # append each run — keeps historical runs
            index=False
        )
        print(f"  Inserted {len(df_to_insert)} row(s) → [{SQL_SCHEMA}].[{table_name}]")

print("\nDone! All metadata written to SQL Server.")

## 9. Quick verification query

In [ ]:
for table_name in TABLE_MAP:
    result = pd.read_sql(
        f"SELECT TOP 5 * FROM [{SQL_SCHEMA}].[{table_name}] ORDER BY id DESC",
        engine
    )
    print(f"\n── {table_name} ──")
    display(result)